In [1]:
# Preliminary packages
%load_ext autoreload
%autoreload 2
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from timeit import default_timer
from functools import reduce
from pathlib import Path
import operator
import math
from time import time
from prettytable import PrettyTable
from scipy.ndimage import distance_transform_edt

from utilities.losses import LpLoss
from utilities.normalizers import GaussianNormalizer
from utilities.plotting import set_size

if torch.cuda.is_available():
    torch.set_default_device('cuda')
    print("Using CUDA as the default device.")
elif torch.backends.mps.is_available():
    torch.set_default_device('mps')
    print("Using MPS as the default device.")
else:
    print("Using CPU as the default device.")

# SETUP PLOTTING FORMATTING
tex_fonts = {
    # Use LaTeX to write all text
    "text.usetex": True,
    "font.family": "times",
    # Use 10pt font in plots, to match 10pt font in document
    "axes.labelsize": 10,
    "font.size": 10,
    # Make the legend/bel fonts a little smaller
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
}

plt.rcParams.update(tex_fonts)

from models.fnoMultiGoal import FNO2dMultiGoal
from models.dafnoMultiGoal import DAFNO2dMultiGoal
from models.deepnormMultiGoal import DEEPNORM2dMultiGoal

# Needed for approximating the SDF solution
from models.fno import FNO2d

torch.cuda.empty_cache()

Using CUDA as the default device.


In [2]:
# Load different set of models for training
savepaths = ["FNOMultiGoal", "DAFNOMultiGoal", "DeepnormMultiGoal", "DeepnormMultiGoal"]

###################
#change inputs to 5
model_classes = [FNO2dMultiGoal(4,1, 8,8,16),\
                 DAFNO2dMultiGoal(4, 8,8, 16),\
                 DEEPNORM2dMultiGoal(4, 8,8,16, in_channels=2),\
                 DEEPNORM2dMultiGoal(4, 8,8,16, in_channels=2)]
###################

# controls whether or not the mask function in DAFNO is applied
mask_func_arr = [lambda a: 1, lambda a: a, lambda a: a,  lambda a: a]

In [3]:
# Load and build normalized torch datasets
dirPath = 'dataset/synthetic/64x64/'
mask = np.load(dirPath + 'mask.npy')
mask = torch.tensor(mask, dtype=torch.float).cuda()
dist_in = np.load(dirPath + 'dist_in.npy')
dist_in = torch.tensor(dist_in, dtype=torch.float).cuda()
goals = np.load(dirPath + 'goal.npy')
goals = torch.tensor(goals, dtype=torch.int).cuda()

##############
#add risk maps
risk_maps = np.load(dirPath + 'risk_maps.npy')
risk_maps = torch.tensor(risk_maps, dtype=torch.float).cuda()
##############

def smooth_chi(mask, dist, smooth_coef):
    return torch.mul(torch.tanh(dist * smooth_coef), (mask - 0.5)) + 0.5
input = smooth_chi(mask, dist_in, 5)
output = np.load(dirPath + 'output.npy')
output = torch.tensor(output, dtype=torch.float).cuda()


nData = len(input)
nTrain = int(nData * 0.8)
nTest = int(nData*0.2)
batch_size=20
mask_train = mask[:nTrain]
mask_test = mask[-nTest:]
goals_train = goals[:nTrain]
goals_test = goals[-nTest:]
chi_train = input[:nTrain]
chi_test = input[-nTest:]
y_train = output[:nTrain]
y_test = output[-nTest:]

######################################
#add risk maps to training and testing
risk_train = risk_maps[:nTrain]
risk_test = risk_maps[-nTest:]
######################################

print(nTest, nTrain)
mask_train = mask_train.reshape(nTrain, input.shape[1], input.shape[2], 1)
mask_test = mask_test.reshape(nTest, input.shape[1], input.shape[2], 1)
goals_train = goals_train.reshape(nTrain, 2)
goals_test = goals_test.reshape(nTest, 2)
chi_train = chi_train.reshape(nTrain, input.shape[1], input.shape[2], 1)
chi_test = chi_test.reshape(nTest, input.shape[1], input.shape[2], 1)

##################
#reshape risk maps
risk_train = risk_train.reshape(nTrain, input.shape[1], input.shape[2], 1)
risk_test = risk_test.reshape(nTest, input.shape[1], input.shape[2], 1)
##################

y_train = y_train.reshape(nTrain, input.shape[1], input.shape[2], 1)
y_test = y_test.reshape(nTest, input.shape[1], input.shape[2], 1)

################################
# concatenate all input channels
X_train = torch.cat((mask_train, chi_train, risk_train), dim=-1) #[B, N, N, 3]
#X_train = X_train.permute(0, 3, 1, 2)

X_test = torch.cat((mask_test, chi_test, risk_test), dim=-1)
#X_test = X_test.permute(0, 3, 1, 2)
################################

chi_combined_train = torch.cat((chi_train, risk_train), dim=-1)  # [B,N,N,2]
chi_combined_test  = torch.cat((chi_test,  risk_test),  dim=-1)  # [B,N,N,2]

###########################
#change to 5 input channels
trainData = DataLoader(TensorDataset(mask_train, chi_combined_train, goals_train, y_train), batch_size=batch_size, shuffle=False, generator=torch.Generator(device='cuda'))
testData = DataLoader(TensorDataset(mask_test, chi_combined_test, goals_test, y_test), batch_size=batch_size, shuffle=False, generator=torch.Generator(device='cuda'))
trainDataSDF = DataLoader(TensorDataset(mask_train, risk_train, dist_in[:nTrain]), batch_size=batch_size, shuffle=True,  generator=torch.Generator(device='cuda'))
testDataSDF = DataLoader(TensorDataset(mask_test, risk_test, dist_in[-nTest:]), batch_size=batch_size, shuffle=True,  generator=torch.Generator(device='cuda'))
###########################



200 800


In [4]:
def count_params(model):
    c = 0
    for p in list(model.parameters()):
        c += reduce(operator.mul,
                    list(p.size()+(2,) if p.is_complex() else p.size()))
    return c

# Train value function model
def train_model(lr, gamma, wd, batch_size, epochs, scheduler_step, mask_func,loss_type, normalizer,\
                model, trainData, testData, print_output, save_file, file_path):
    # Setup optimizers
    loss_func = LpLoss(d=2, p=2)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=scheduler_step, gamma=gamma)

    # Setup data collection
    training_losses = []
    test_losses = []
    train_time_arr = []
    best_model_weights = None
    best_training_loss = np.inf
    best_test_loss = np.inf
    best_epoch = None

    # Print header
    print("Training model", file_path)
    print("Epoch", "Epoch time", "Train loss", "Test loss")

    for ep in range(epochs):
        model.train()

        # Training
        t1 = default_timer()
        train_loss = 0
        samples_train = 0
        mask_train, chi_combined_train, goals_train, y_train
        for mask, chi, goals, y in trainData: ###X
            mask.cuda(), chi.cuda(), y.cuda()###

            # Setup training
            optimizer.zero_grad()

            # Forward pass
            out = model(chi, goals)
            if normalizer is not None:
                out = normalizer.decode(out) * mask_func(mask)
                y = normalizer.decode(y) * mask_func(mask)
            else:
                out = out*mask_func(mask)
            # PINN loss
            if loss_type == "PINN":
                grad_out = torch.linalg.vector_norm(torch.stack(list(torch.gradient(out.reshape(out.shape[0], out.shape[1], out.shape[2]), dim=[1, 2])), dim=0), dim=0)
                grad_y = torch.linalg.vector_norm(torch.stack(list(torch.gradient(y.reshape(out.shape[0], out.shape[1], out.shape[2]), dim=[1, 2])), dim=0), dim=0)
                # set boundary gradients to zero
                grad_mask = grad_y.clone()
                grad_mask[grad_mask >= 1.01] = 0
               
                loss =loss_func(out.view(y.shape[0], y.shape[1], y.shape[2]), y.view(y.shape[0], y.shape[1], y.shape[2])) + \
                0.05*loss_func((grad_out*grad_mask).view(batch_size, y.shape[1], y.shape[2]),grad_mask.view((batch_size, y.shape[1], y.shape[2])))
            else:
                loss =loss_func(out.view(y.shape[0], y.shape[1], y.shape[2]), y.view(y.shape[0], y.shape[1], y.shape[2])) 
            train_loss += loss.item()
            samples_train += y.shape[0]

            # Backprop
            loss.backward()
            optimizer.step()       
        scheduler.step()
        model.eval()
        test_loss = 0
        loss = 0
        t2 = default_timer()
        # Testing
        samples_test = 0
        with torch.no_grad():
            for mask, chi, goals, y in testData:
                out = model(chi, goals)
                if normalizer is not None:
                    out = normalizer.decode(out) * mask_func(mask)
                else:
                    out = out*mask_func(mask)
                # PINN loss
                if loss_type == "PINN":
                    grad_out = torch.linalg.vector_norm(torch.stack(list(torch.gradient(out.reshape(out.shape[0], out.shape[1], out.shape[2]), dim=[1, 2])), dim=0), dim=0)
                    grad_y = torch.linalg.vector_norm(torch.stack(list(torch.gradient(y.reshape(out.shape[0], out.shape[1], out.shape[2]), dim=[1, 2])), dim=0), dim=0)
                    # set boundary gradients to zero
                    grad_mask = grad_y.clone()
                    grad_mask[grad_mask >= 1.01] = 0
                   
                    loss =loss_func(out.view(y.shape[0], y.shape[1], y.shape[2]), y.view(y.shape[0], y.shape[1], y.shape[2])) + \
                    0.05*loss_func((grad_out*grad_mask).view(batch_size, y.shape[1], y.shape[2]),grad_mask.view((batch_size, y.shape[1], y.shape[2])))
                else:
                    loss =loss_func(out.view(y.shape[0], y.shape[1], y.shape[2]), y.view(y.shape[0], y.shape[1], y.shape[2])) 
                test_loss += loss.item()
                samples_test += y.shape[0]
        # Collect data and setup best model
        train_loss /= samples_train
        test_loss /= samples_test
        training_losses.append(train_loss)
        test_losses.append(test_loss)
        if test_loss < best_test_loss:
            best_test_loss = test_loss
            best_training_loss = train_loss
            best_epoch = ep   
            best_model_weights = model.state_dict()

        t2 = default_timer()
        train_time_arr.append(t2-t1)
        if ep%10 == 0 and print_output:
            print(ep, t2-t1, train_loss, test_loss)

    if save_file:
        Path(file_path).mkdir(parents=True, exist_ok=True)
        np.savetxt(file_path + "/training_losses.txt", training_losses)
        np.savetxt(file_path + "/test_losses.txt", test_losses)
        np.savetxt(file_path + "/training_times.txt", train_time_arr)
        torch.save(best_model_weights, file_path + "/best_model.pt")
        f = open(file_path + "/aggregate_results.txt", "w")
        f.write("ModelParameters: " + str(count_params(model)) + "\n")
        f.write("BestL2TrainingError: " + str(best_training_loss) + "\n")
        f.write("BestL2TestingError: " + str(best_test_loss) + "\n")
        f.write("BestEpoch: " + str(best_epoch) + "\n")
        f.write("TotalTrainingTime: " + str(np.sum(train_time_arr)) + "\n")
        f.close()

def count_params(model):
    c = 0
    for p in list(model.parameters()):
        c += reduce(operator.mul,
                    list(p.size()+(2,) if p.is_complex() else p.size()))
    return c

# Train SDF function model
def train_model_SDF(lr, gamma, wd, batch_size, epochs, scheduler_step, \
                model, trainData, testData, print_output, save_file, file_path):
    # Setup optimizers
    loss_func = LpLoss(d=2, p=2)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=scheduler_step, gamma=gamma)

    # Setup data collection
    training_losses = []
    test_losses = []
    train_time_arr = []
    best_model_weights = None
    best_training_loss = np.inf
    best_test_loss = np.inf
    best_epoch = None

    # Print header
    print("Training model", file_path)
    print("Epoch", "Epoch time", "Train loss", "Test loss")

    for ep in range(epochs):
        model.train()

        # Training
        t1 = default_timer()
        train_loss = 0
        samples_train = 0
        for mask, y in testData:
            mask.cuda(), y.cuda()
            # Setup training
            optimizer.zero_grad()
    
            # Forward pass
            out = model(mask)

            loss =loss_func(out.view(y.shape[0], y.shape[1], y.shape[2]), y.view(y.shape[0], y.shape[1], y.shape[2])) 
            train_loss += loss.item()
            samples_train += y.shape[0]

            # Backprop
            loss.backward()
            optimizer.step()       
        scheduler.step()
        model.eval()
        test_loss = 0
        loss = 0
        t2 = default_timer()
        # Testing
        samples_test = 0
        with torch.no_grad():
            for mask, y in testData:
                mask.cuda(), y.cuda()
                # Setup training
                optimizer.zero_grad()
    
                # Forward pass
                out = model(mask)
                
                loss =loss_func(out.view(y.shape[0], y.shape[1], y.shape[2]), y.view(y.shape[0], y.shape[1], y.shape[2])) 
                test_loss += loss.item()
                samples_test += y.shape[0]
        # Collect data and setup best model
        train_loss /= samples_train
        test_loss /= samples_test
        training_losses.append(train_loss)
        test_losses.append(test_loss)
        if test_loss < best_test_loss:
            best_test_loss = test_loss
            best_training_loss = train_loss
            best_epoch = ep   
            best_model_weights = model.state_dict()

        t2 = default_timer()
        train_time_arr.append(t2-t1)
        if ep%10 == 0 and print_output:
            print(ep, t2-t1, train_loss, test_loss)

    if save_file:
        Path(file_path).mkdir(parents=True, exist_ok=True)
        np.savetxt(file_path + "/training_losses.txt", training_losses)
        np.savetxt(file_path + "/test_losses.txt", test_losses)
        np.savetxt(file_path + "/training_times.txt", train_time_arr)
        torch.save(best_model_weights, file_path + "/best_model.pt")
        f = open(file_path + "/aggregate_results.txt", "w")
        f.write("ModelParameters: " + str(count_params(model)) + "\n")
        f.write("BestL2TrainingError: " + str(best_training_loss) + "\n")
        f.write("BestL2TestingError: " + str(best_test_loss) + "\n")
        f.write("BestEpoch: " + str(best_epoch) + "\n")
        f.write("TotalTrainingTime: " + str(np.sum(train_time_arr)) + "\n")
        f.close()

In [ ]:
# Trains the 4 different models. The ordering is FNO, DAFNO, PNO w reg loss, and PNO w PINN loss. 
# Load different set of models for training
savepaths = ["FNOMultiGoal", "DAFNOMultiGoal", "DeepnormMultiGoal", "DeepnormMultiGoal"]

model_classes = [FNO2dMultiGoal(4,1, 8,8,16),\
                 DAFNO2dMultiGoal(4, 8,8, 16),\
                 DEEPNORM2dMultiGoal(4, 8,8,16, in_channels=2),\
                 DEEPNORM2dMultiGoal(4, 8,8,16, in_channels=2)]

# controls whether or not the mask function in DAFNO is applied
mask_func_arr = [lambda a: 1, lambda a: a, lambda a: a,  lambda a: a]
loss_type_arr = ["L2", "L2", "L2", "PINN"]
model_save_paths = ["FNO", "DAFNO", "PNO", "PNOwPINN"]


for i in range(3, len(savepaths)):
        train_model(lr=1e-3, gamma=0.5, wd=1e-5, batch_size=20, epochs=500, scheduler_step=50, model=model_classes[i],\
                    mask_func=mask_func_arr[i], loss_type=loss_type_arr[i], normalizer=None,\
                    trainData=trainData, testData=testData,\
                    print_output=True, save_file=True, file_path="./results/" + model_save_paths[i])

Training model ./results/PNOwPINN
Epoch Epoch time Train loss Test loss
0 2.3124775942415 1.047266321182251 1.0355904865264893
10 1.7443728670477867 0.4409965693950653 0.39668506383895874
20 1.7842082767747343 0.23490425169467927 0.17258619427680968
30 1.7311338121071458 0.20609337955713272 0.15536213159561157
40 1.787301640957594 0.2044144156575203 0.15595305919647218
50 1.7199519481509924 0.20626203387975692 0.1617158079147339
60 1.7789552542380989 0.187563519179821 0.1509249234199524
70 1.7110854890197515 0.1834460535645485 0.14885419249534607
80 1.7122119660489261 0.17969225019216536 0.14771233439445497
90 1.789753579068929 0.17673418372869493 0.147086021900177
100 1.7601183583028615 0.17409773796796799 0.14619511365890503
110 1.783449687063694 0.1693540221452713 0.1461195695400238
120 1.7563236276619136 0.16680962651968 0.14623336672782897
130 1.8208987540565431 0.1640662357211113 0.14642011642456054
140 1.7774186809547246 0.16229162871837616 0.14642390847206116
150 1.743721662089